循环对所有组做线性回归

In [1]:
import csv
from pathlib import Path
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from scipy import sparse
import harmonypy as hm

In [2]:
def linreg_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]; y = y[ok]
    n = len(x)
    if n < 2:
        return np.nan, np.nan, np.nan, np.nan

    xm = x.mean()
    ym = y.mean()
    dx = x - xm
    dy = y - ym

    varx = np.sum(dx * dx)
    if varx == 0:
        return np.nan, np.nan, np.nan, np.nan

    slope = np.sum(dx * dy) / varx
    intercept = ym - slope * xm

    yhat = slope * x + intercept
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - ym) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

    # y=0 时 x 截距：x0 = -b/a；取绝对值
    x0_abs = np.abs(-intercept / slope) if slope != 0 else np.nan
    return slope, intercept, r2, x0_abs

In [3]:
def build_linreg_result(df):
    """
    df 的行名格式：
    ctrl1_0_cluster0 / dn1_15_cluster2 这类
    列名为 m/z
    """
    tmp = df.copy()

    # 解析行名：样本号、时间、cluster
    parts = tmp.index.to_series().str.extract(
        r"^([A-Za-z]+\d+)_(\d+)_cluster(\d+)$"
    )

    tmp["_sample"] = parts[0]
    tmp["_time"] = parts[1].astype(int)
    tmp["_cluster"] = parts[2].astype(int)

    # m/z 列
    mz_cols = [c for c in tmp.columns if c not in ["_sample", "_time", "_cluster"]]
    mz_cols = sorted(mz_cols, key=lambda x: float(x))

    result_rows = []

    # 按 sample + cluster 分组
    for (sample, cluster), sub in tmp.groupby(["_sample", "_cluster"], sort=False):
        sub = sub.sort_values("_time")

        # 这里 x 实际上就是 [0,15,30,45,60]
        x = sub["_time"].to_numpy(dtype=float)

        for mz in mz_cols:
            y = sub[mz].to_numpy(dtype=float)
            slope, intercept, r2, x0_abs = linreg_stats(x, y)

            result_rows.append({
                "sample": sample,
                "mz": float(mz),
                "cluster": int(cluster),
                "slope": slope,
                "intercept": intercept,
                "r2": r2,
                "x0_abs": x0_abs
            })

    result_df = pd.DataFrame(result_rows)

    # 排序：先 sample，再 cluster，再 mz
    parts2 = result_df["sample"].str.extract(r"^([A-Za-z]+)(\d+)$")
    result_df["_prefix"] = parts2[0]
    result_df["_sample_num"] = parts2[1].astype(int)

    result_df = result_df.sort_values(
        by=["_prefix", "_sample_num", "cluster", "mz"],
        kind="stable"
    ).drop(columns=["_prefix", "_sample_num"])

    result_df = result_df.reset_index(drop=True)
    return result_df

In [4]:
# ===== 参数区：以后换组主要改这里 =====
PARAM_BASE_PATH = Path("/p2/zulab/jtian/data/SA/06_calculateConcentration/CAST_Kmeans_seperate3")
PARAM_INPUT_BASE_PATH = Path("/p2/zulab/jtian/data/SA/06_calculateConcentration/MetaboliteIntensity/scilsPeakFilter")
PARAM_CLUSTER_CSV = Path("/p2/zulab/jtian/data/SA/R-Integration/Integration2_downsample_5000/cluster.csv")

def load_integration_cluster_labels(cluster_csv):
    """Load Seurat integration clusters and convert Spot IDs to notebook pixel IDs."""
    cluster_df = pd.read_csv(cluster_csv, index_col=0)
    if cluster_df.shape[1] != 1:
        raise ValueError(f"Expected one cluster column in {cluster_csv}, got {cluster_df.shape[1]}")

    labels = cluster_df.iloc[:, 0].astype(int).rename("cluster")
    labels.index = labels.index.to_series().str.replace(
        r"_Spot_(\d+)$",
        r"_pixel\1",
        regex=True,
    )

    parts = labels.index.to_series().str.extract(r"^([A-Za-z]+\d+)_(\d+)_pixel(\d+)$")
    if parts.isna().any().any():
        bad = labels.index[parts.isna().any(axis=1)][:5].tolist()
        raise ValueError(f"Unexpected cluster index format in {cluster_csv}: {bad}")

    return labels

integration_cluster_labels = load_integration_cluster_labels(PARAM_CLUSTER_CSV)
PARAM_N_CLUSTERS = int(integration_cluster_labels.nunique())
print(
    f"Loaded {len(integration_cluster_labels)} integration cluster labels "
    f"from {PARAM_CLUSTER_CSV}; n_clusters={PARAM_N_CLUSTERS}"
)
PARAM_GROUPS = ["ctrl1","ctrl2","ctrl3", "dn1", "dn2", "dn3"]
PARAM_TIME_POINTS = ["0", "15", "30", "45", "60"]
PARAM_R2_THRESHOLD = 0.9
PARAM_CLEAN_INPUT_CSV = True


Loaded 149999 integration cluster labels from /p2/zulab/jtian/data/SA/R-Integration/Integration2_downsample_5000/cluster.csv; n_clusters=13


In [5]:
# ===== 直接运行版：不使用 def；换组只改上面的 PARAM_GROUPS =====
cast_results = {}

for GROUP in PARAM_GROUPS:
    print(f"\n===== Running {GROUP} =====")
    data_dir = Path(PARAM_BASE_PATH) / GROUP
    data_dir.mkdir(parents=True, exist_ok=True)
    input_data_dir = Path(PARAM_INPUT_BASE_PATH) / GROUP
    
    if PARAM_CLEAN_INPUT_CSV:
        for tp in PARAM_TIME_POINTS:
            csv_file = input_data_dir / f"{tp}.csv"
            print(f"\nReading file: {csv_file.name}")
            df = pd.read_csv(csv_file, sep=";", comment="#")
            print(df.iloc[:5, :5])
            df.to_csv(csv_file, sep=";", index=False)
            print(f"Cleaned and overwritten: {csv_file.name}")

    group_list = []
    for tp in PARAM_TIME_POINTS:
        csv_file = input_data_dir / f"{tp}.csv"
        df = pd.read_csv(csv_file, sep=";", index_col=0, header=0)
        df_t = df.T.copy()
        duplicated_mz = df_t.columns[df_t.columns.duplicated()].unique().tolist()
        if duplicated_mz:
            print(
                f"{GROUP} {tp}.csv duplicated mz: {len(duplicated_mz)}; "
                f"examples: {duplicated_mz[:5]}"
            )
        else:
            print(f"{GROUP} {tp}.csv duplicated mz: 0")
        
        df_t = df_t.loc[:, ~df_t.columns.duplicated()]  # 去掉包里筛出来的重复 mz
        df_t.index = [f"{GROUP}_{tp}_pixel{i + 1}" for i in range(df_t.shape[0])]
        group_list.append(df_t)

    group_merged = pd.concat(group_list, axis=0)
    print(f"{GROUP}_merged shape:", group_merged.shape)
    print(group_merged.head())

    group_cluster = integration_cluster_labels.loc[
        integration_cluster_labels.index.to_series().str.startswith(f"{GROUP}_")
    ]
    matched_index = group_merged.index.intersection(group_cluster.index)
    missing_labels = group_merged.shape[0] - len(matched_index)
    extra_labels = len(group_cluster) - len(matched_index)
    print(
        f"{GROUP}: matched {len(matched_index)} integration labels; "
        f"dropped {missing_labels} unlabeled pixels; unused labels {extra_labels}"
    )
    if matched_index.empty:
        raise ValueError(f"{GROUP}: no pixels matched integration cluster labels")

    group_merged = group_merged.loc[matched_index].copy()
    group_merged.insert(0, "cluster", group_cluster.loc[matched_index].to_numpy())

    group_mz_cols = [c for c in group_merged.columns if c != "cluster"]
    group_mz_cols = sorted(group_mz_cols, key=lambda x: float(x))
    group_merged = group_merged[["cluster"] + group_mz_cols]
    group_merged.to_csv(data_dir / f"{GROUP}Intensity_merged.csv", sep=";")

    group_tmp = group_merged.copy()
    group_tmp["group"] = group_tmp.index.to_series().str.replace(r"_pixel\d+$", "", regex=True)

    group_cluster_mean = group_tmp.groupby(
        ["group", "cluster"],
        sort=False,
    )[group_mz_cols].mean()

    group_cluster_mean.index = [
        f"{sample_group}_cluster{cluster_id}"
        for sample_group, cluster_id in group_cluster_mean.index
    ]

    group_sort_df = group_cluster_mean.copy()
    group_parts = group_sort_df.index.to_series().str.extract(
        r"^([A-Za-z]+)(\d+)_(\d+)_cluster(\d+)$"
    )

    group_sort_df["_prefix"] = group_parts[0]
    group_sort_df["_sample_num"] = group_parts[1].astype(int)
    group_sort_df["_time"] = group_parts[2].astype(int)
    group_sort_df["_cluster"] = group_parts[3].astype(int)

    group_sort_df = group_sort_df.sort_values(
        by=["_prefix", "_sample_num", "_cluster", "_time"],#"_prefix"按字母顺序排序
        kind="stable",
    )

    group_cluster_mean = group_sort_df.drop(
        columns=["_prefix", "_sample_num", "_time", "_cluster"]
    )
    group_cluster_mean.to_csv(data_dir / f"{GROUP}_cluster_mean.csv", sep=";")
    print(group_cluster_mean.iloc[:10, :5])

    group_linreg_result = build_linreg_result(group_cluster_mean)
    group_linreg_result.to_csv(data_dir / f"{GROUP}_linreg_result.csv", index=False)
    print(group_linreg_result.head())

    group_r2_mean = (
        group_linreg_result
        .groupby(["sample", "mz"], as_index=False)["r2"]
        .mean()
        .rename(columns={"r2": "mean_r2"})
    )

    group_good = group_r2_mean[group_r2_mean["mean_r2"] >= PARAM_R2_THRESHOLD].copy()

    group_count = (
        group_good
        .groupby("sample", as_index=False)
        .size()
        .rename(columns={"size": f"n_metabolites_r2_ge_{PARAM_R2_THRESHOLD}"})
    )

    group_good.to_csv(data_dir / f"{GROUP}_good_r2_ge_{PARAM_R2_THRESHOLD}.csv", index=False)
    group_count.to_csv(data_dir / f"{GROUP}_count_r2_ge_{PARAM_R2_THRESHOLD}.csv", index=False)

    print(f"\n{GROUP}_good:")
    print(group_good.head())
    print(f"\n{GROUP}_count:")
    print(group_count)

    cast_results[GROUP] = {
        "merged": group_merged,
        "cluster_mean": group_cluster_mean,
        "linreg_result": group_linreg_result,
        "r2_mean": group_r2_mean,
        "good": group_good,
        "count": group_count,
    }



===== Running ctrl1 =====

Reading file: 0.csv
         m/z    Spot 1    Spot 2    Spot 3    Spot 4
0  57.034588  0.000000  0.000000  0.000000  0.146225
1  58.029870  0.153058  0.000000  0.230207  0.000000
2  59.013891  0.000000  0.224234  0.000000  0.000000
3  67.019106  0.000000  0.368060  0.000000  0.440416
4  68.013943  0.208158  0.181199  0.237401  0.319142
Cleaned and overwritten: 0.csv

Reading file: 15.csv
         m/z  Spot 26477  Spot 26478  Spot 26479  Spot 26480
0  57.034588    0.362216    0.550330    0.000000    0.221399
1  58.029870    0.096241    0.851077    0.515851    0.586297
2  59.013891    0.000000    4.225184    3.650091    5.585155
3  67.019106    0.000000    1.908059    0.658155    0.000000
4  68.013943    0.943162    0.480447    0.000000    0.703556
Cleaned and overwritten: 15.csv

Reading file: 30.csv
         m/z  Spot 57458  Spot 57459  Spot 57460  Spot 57461
0  57.034588    0.000000    0.000000    0.582124    0.000000
1  58.029870    1.619663    0.353508   

KeyboardInterrupt: 

In [ ]:
# ===== 循环画出 PARAM_GROUPS 中所有 group 的 cluster count heatmap =====
PARAM_HEATMAP_H5AD = Path(
    "/p2/zulab/jtian/data/SA/06_calculateConcentration/output_CAST_Kmeans_seperate/adata30.h5ad"
)
coords_df = sc.read_h5ad(PARAM_HEATMAP_H5AD).obs[["sample", "x", "y"]].copy()
coords_df.index = coords_df.index.astype(str)
count_col = f"n_metabolites_r2_ge_{PARAM_R2_THRESHOLD}"
cmap = plt.cm.viridis

for GROUP in PARAM_GROUPS:
    data_dir = Path(PARAM_BASE_PATH) / GROUP
    merged_df = cast_results[GROUP]["merged"]
    linreg_df = cast_results[GROUP]["linreg_result"]

    cluster_count_df = (
        linreg_df.loc[linreg_df["r2"] >= PARAM_R2_THRESHOLD]
        .groupby(["sample", "cluster"], as_index=False)["mz"]
        .nunique()
        .rename(columns={"mz": count_col})
    )
    cluster_count_df.to_csv(
        data_dir / f"{GROUP}_cluster_count_r2_ge_{PARAM_R2_THRESHOLD}.csv",
        index=False,
    )

    cluster_to_count = cluster_count_df.set_index("cluster")[count_col].to_dict()
    sample_counts = np.array(list(cluster_to_count.values()), dtype=float)
    if len(sample_counts) == 0:
        vmin, vmax = 0, 1
    else:
        vmin = np.percentile(sample_counts, 5)
        vmax = np.percentile(sample_counts, 95)
        if vmin == vmax:
            vmin, vmax = sample_counts.min(), sample_counts.max()
        if vmin == vmax:
            vmax = vmin + 1

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    fig, axes = plt.subplots(
        1,
        len(PARAM_TIME_POINTS),
        figsize=(5 * len(PARAM_TIME_POINTS), 6),
        constrained_layout=True,
    )

    for ax, tp in zip(np.ravel(axes), PARAM_TIME_POINTS):
        sample_key = f"{GROUP}_{tp}"
        labels = merged_df.loc[
            merged_df.index.to_series().str.startswith(f"{sample_key}_pixel"),
            "cluster",
        ]
        coords_sub = coords_df.loc[labels.index, ["x", "y"]].copy()

        if len(coords_sub) != len(labels):
            raise ValueError(
                f"{sample_key}: coords ({len(coords_sub)}) and labels ({len(labels)}) do not match"
            )

        coords_sub[count_col] = labels.map(cluster_to_count).fillna(0).to_numpy()
        sc_plot = ax.scatter(
            coords_sub["x"],
            coords_sub["y"],
            c=coords_sub[count_col],
            cmap=cmap,
            norm=norm,
            s=3,
            edgecolors="none",
            rasterized=True,
        )
        ax.set_title(sample_key, fontsize=12)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])

    fig.colorbar(sc_plot, ax=np.ravel(axes), fraction=0.03, pad=0.02, extend="both").set_label(
        f"Metabolites with r2 >= {PARAM_R2_THRESHOLD}\n(sample-wise 5%-95% range)"
    )

    legend_handles = []
    for cluster in sorted(range(PARAM_N_CLUSTERS), key=lambda x: (-int(cluster_to_count.get(x, 0)), x)):
        count = int(cluster_to_count.get(cluster, 0))
        color = cmap(norm(min(max(count, vmin), vmax)))
        legend_handles.append(
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                label=f"cluster {cluster}: {count}",
                markerfacecolor=color,
                markersize=8,
            )
        )

    fig.legend(
        handles=legend_handles,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=False,
        title=f"Cluster counts\nq5={vmin:.1f}, q95={vmax:.1f}",
    )
    fig.suptitle(f"{GROUP}: KMeans cluster heatmap by metabolite count", fontsize=14)

    fig.savefig(
        data_dir / f"{GROUP}_cluster_r2_ge_{PARAM_R2_THRESHOLD:.1f}_heatmap_q5_q95_bluepink.png",
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        data_dir / f"{GROUP}_cluster_r2_ge_{PARAM_R2_THRESHOLD:.1f}_heatmap_q5_q95_bluepink.pdf",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


In [ ]:
# ===== 不保存结果：单独运行版，用不同时间点拟合标准曲线并计算 R2 =====
# 这个 cell 不依赖前面的 cast_results；可以直接运行。
# 优先读取已经保存的 *_cluster_mean.csv；如果文件不存在，就从原始 intensity csv 和 Integration2 的 cluster.csv 临时生成。
# 不保存任何结果，只生成 linreg_time_variant_results 变量并打印统计。

from pathlib import Path
import math
import numpy as np
import pandas as pd

PARAM_BASE_PATH = Path("/p2/zulab/jtian/data/SA/06_calculateConcentration/CAST_Kmeans_seperate3")
PARAM_INPUT_BASE_PATH = Path("/p2/zulab/jtian/data/SA/06_calculateConcentration/MetaboliteIntensity/scilsPeakFilter")
PARAM_CLUSTER_CSV = Path("/p2/zulab/jtian/data/SA/R-Integration/Integration2_downsample_5000/cluster.csv")

def load_integration_cluster_labels(cluster_csv):
    """Load Seurat integration clusters and convert Spot IDs to notebook pixel IDs."""
    cluster_df = pd.read_csv(cluster_csv, index_col=0)
    if cluster_df.shape[1] != 1:
        raise ValueError(f"Expected one cluster column in {cluster_csv}, got {cluster_df.shape[1]}")

    labels = cluster_df.iloc[:, 0].astype(int).rename("cluster")
    labels.index = labels.index.to_series().str.replace(
        r"_Spot_(\d+)$",
        r"_pixel\1",
        regex=True,
    )

    parts = labels.index.to_series().str.extract(r"^([A-Za-z]+\d+)_(\d+)_pixel(\d+)$")
    if parts.isna().any().any():
        bad = labels.index[parts.isna().any(axis=1)][:5].tolist()
        raise ValueError(f"Unexpected cluster index format in {cluster_csv}: {bad}")

    return labels

integration_cluster_labels = load_integration_cluster_labels(PARAM_CLUSTER_CSV)
PARAM_N_CLUSTERS = int(integration_cluster_labels.nunique())
print(
    f"Loaded {len(integration_cluster_labels)} integration cluster labels "
    f"from {PARAM_CLUSTER_CSV}; n_clusters={PARAM_N_CLUSTERS}"
)
PARAM_GROUPS = ["ctrl1", "ctrl2", "ctrl3", "dn1", "dn2", "dn3"]
PARAM_TIME_POINTS = ["0", "15", "30", "45", "60"]
PARAM_R2_THRESHOLD = 0.9
PARAM_LINREG_TIME_VARIANTS = {
    "fit_0_15_30_45": [0, 15, 30, 45],
    "fit_0_15_30": [0, 15, 30],
}


def linreg_stats_standalone(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]
    n = len(x)
    if n < 2:
        return np.nan, np.nan, np.nan, np.nan

    xm = x.mean()
    ym = y.mean()
    dx = x - xm
    dy = y - ym

    varx = np.sum(dx * dx)
    if varx == 0:
        return np.nan, np.nan, np.nan, np.nan

    slope = np.sum(dx * dy) / varx
    intercept = ym - slope * xm

    yhat = slope * x + intercept
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - ym) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan
    x0_abs = np.abs(-intercept / slope) if slope != 0 else np.nan
    return slope, intercept, r2, x0_abs


def load_or_build_group_cluster_mean(group):
    cluster_mean_file = PARAM_BASE_PATH / group / f"{group}_cluster_mean.csv"
    if cluster_mean_file.exists():
        print(f"{group}: read existing {cluster_mean_file.name}")
        return pd.read_csv(cluster_mean_file, sep=";", index_col=0)

    print(f"{group}: {cluster_mean_file.name} not found, build from raw csv temporarily")
    input_data_dir = PARAM_INPUT_BASE_PATH / group
    group_list = []

    for tp in PARAM_TIME_POINTS:
        csv_file = input_data_dir / f"{tp}.csv"
        df = pd.read_csv(csv_file, sep=";", index_col=0, header=0)
        df_t = df.T.copy()
        df_t = df_t.loc[:, ~df_t.columns.duplicated()]
        df_t.index = [f"{group}_{tp}_pixel{i + 1}" for i in range(df_t.shape[0])]
        group_list.append(df_t)

    group_merged = pd.concat(group_list, axis=0)
    group_cluster = integration_cluster_labels.loc[
        integration_cluster_labels.index.to_series().str.startswith(f"{group}_")
    ]
    matched_index = group_merged.index.intersection(group_cluster.index)
    missing_labels = group_merged.shape[0] - len(matched_index)
    extra_labels = len(group_cluster) - len(matched_index)
    print(
        f"{group}: matched {len(matched_index)} integration labels; "
        f"dropped {missing_labels} unlabeled pixels; unused labels {extra_labels}"
    )
    if matched_index.empty:
        raise ValueError(f"{group}: no pixels matched integration cluster labels")

    group_merged = group_merged.loc[matched_index].copy()
    group_merged.insert(0, "cluster", group_cluster.loc[matched_index].to_numpy())

    group_mz_cols = [c for c in group_merged.columns if c != "cluster"]
    group_mz_cols = sorted(group_mz_cols, key=lambda x: float(x))
    group_merged = group_merged[["cluster"] + group_mz_cols]

    group_tmp = group_merged.copy()
    group_tmp["group"] = group_tmp.index.to_series().str.replace(
        r"_pixel\d+$", "", regex=True
    )

    group_cluster_mean = group_tmp.groupby(
        ["group", "cluster"],
        sort=False,
    )[group_mz_cols].mean()

    group_cluster_mean.index = [
        f"{sample_group}_cluster{cluster_id}"
        for sample_group, cluster_id in group_cluster_mean.index
    ]

    group_sort_df = group_cluster_mean.copy()
    group_parts = group_sort_df.index.to_series().str.extract(
        r"^([A-Za-z]+)(\d+)_(\d+)_cluster(\d+)$"
    )
    group_sort_df["_prefix"] = group_parts[0]
    group_sort_df["_sample_num"] = group_parts[1].astype(int)
    group_sort_df["_time"] = group_parts[2].astype(int)
    group_sort_df["_cluster"] = group_parts[3].astype(int)
    group_sort_df = group_sort_df.sort_values(
        by=["_prefix", "_sample_num", "_cluster", "_time"],
        kind="stable",
    )
    return group_sort_df.drop(columns=["_prefix", "_sample_num", "_time", "_cluster"])


def build_linreg_result_for_time_points(df, fit_time_points):
    tmp = df.copy()
    parts = tmp.index.to_series().str.extract(
        r"^([A-Za-z]+\d+)_(\d+)_cluster(\d+)$"
    )
    tmp["_sample"] = parts[0]
    tmp["_time"] = parts[1].astype(int)
    tmp["_cluster"] = parts[2].astype(int)
    tmp = tmp[tmp["_time"].isin(fit_time_points)].copy()

    mz_cols = [c for c in tmp.columns if c not in ["_sample", "_time", "_cluster"]]
    mz_cols = sorted(mz_cols, key=lambda x: float(x))
    result_rows = []

    for (sample, cluster), sub in tmp.groupby(["_sample", "_cluster"], sort=False):
        sub = sub.sort_values("_time")
        x = sub["_time"].to_numpy(dtype=float)

        for mz in mz_cols:
            y = sub[mz].to_numpy(dtype=float)
            slope, intercept, r2, x0_abs = linreg_stats_standalone(x, y)
            result_rows.append({
                "sample": sample,
                "mz": float(mz),
                "cluster": int(cluster),
                "slope": slope,
                "intercept": intercept,
                "r2": r2,
                "x0_abs": x0_abs,
            })

    result_df = pd.DataFrame(result_rows)
    parts2 = result_df["sample"].str.extract(r"^([A-Za-z]+)(\d+)$")
    result_df["_prefix"] = parts2[0]
    result_df["_sample_num"] = parts2[1].astype(int)
    result_df = result_df.sort_values(
        by=["_prefix", "_sample_num", "cluster", "mz"],
        kind="stable",
    ).drop(columns=["_prefix", "_sample_num"])
    return result_df.reset_index(drop=True)


def mz_key_floor_2dec(mz):
    # 小数点后只取前两位，不四舍五入；例如 503.264514 -> 503.26
    return math.floor(float(mz) * 100) / 100


group_cluster_mean_cache = {
    group: load_or_build_group_cluster_mean(group)
    for group in PARAM_GROUPS
}

linreg_time_variant_results = {}

for variant_name, fit_time_points in PARAM_LINREG_TIME_VARIANTS.items():
    print(f"\n===== {variant_name}: use time points {fit_time_points} =====")
    variant_results = {}
    all_clusters_good_list = []

    for group in PARAM_GROUPS:
        group_cluster_mean = group_cluster_mean_cache[group]
        group_linreg_result = build_linreg_result_for_time_points(
            group_cluster_mean,
            fit_time_points,
        )

        group_r2_mean = (
            group_linreg_result
            .groupby(["sample", "mz"], as_index=False)["r2"]
            .mean()
            .rename(columns={"r2": "mean_r2"})
        )
        group_good = group_r2_mean[
            group_r2_mean["mean_r2"] >= PARAM_R2_THRESHOLD
        ].copy()

        group_all_cluster_summary = (
            group_linreg_result
            .groupby(["sample", "mz"], as_index=False)
            .agg(
                n_clusters=("cluster", "nunique"),
                n_rows=("r2", "size"),
                min_r2=("r2", "min"),
                max_r2=("r2", "max"),
            )
        )
        group_all_clusters_good = group_all_cluster_summary[
            (group_all_cluster_summary["n_clusters"] == PARAM_N_CLUSTERS)
            & (group_all_cluster_summary["n_rows"] == PARAM_N_CLUSTERS)
            & (group_all_cluster_summary["min_r2"] >= PARAM_R2_THRESHOLD)
        ].copy()
        group_all_clusters_good["group"] = group
        group_all_clusters_good["mz_key_2dec"] = group_all_clusters_good["mz"].map(
            mz_key_floor_2dec
        )
        all_clusters_good_list.append(group_all_clusters_good)

        variant_results[group] = {
            "linreg_result": group_linreg_result,
            "r2_mean": group_r2_mean,
            "group_good": group_good,
            "all_clusters_summary": group_all_cluster_summary,
            "all_clusters_good": group_all_clusters_good,
        }

        print(
            f"{group}: group_good(mean R2 >= {PARAM_R2_THRESHOLD}) = {len(group_good)}, "
            f"all {PARAM_N_CLUSTERS} clusters R2 >= {PARAM_R2_THRESHOLD} = {len(group_all_clusters_good)}"
        )

    all_clusters_good_df = pd.concat(all_clusters_good_list, ignore_index=True)
    all6_keys = (
        all_clusters_good_df
        .groupby("mz_key_2dec")
        .agg(
            n_groups=("group", "nunique"),
            global_min_r2=("min_r2", "min"),
            global_max_r2=("max_r2", "max"),
            n_group_mz_records=("mz", "size"),
        )
        .reset_index()
    )
    all6_keys = all6_keys[all6_keys["n_groups"] == len(PARAM_GROUPS)].copy()
    all6_keys = all6_keys.sort_values("mz_key_2dec").reset_index(drop=True)

    all6_detail = all_clusters_good_df[
        all_clusters_good_df["mz_key_2dec"].isin(all6_keys["mz_key_2dec"])
    ].sort_values(["mz_key_2dec", "group", "mz"])

    print(
        f"\n{variant_name}: 6 groups all have all-{PARAM_N_CLUSTERS}-cluster R2 >= "
        f"{PARAM_R2_THRESHOLD} metabolite keys = {len(all6_keys)}"
    )
    if len(all6_keys) > 0:
        print(all6_keys[["mz_key_2dec", "global_min_r2", "global_max_r2"]].to_string(index=False))
        print("\nMatched original mz by group:")
        print(
            all6_detail[["mz_key_2dec", "group", "mz", "min_r2", "max_r2"]]
            .to_string(index=False)
        )

    variant_results["all_clusters_good_df"] = all_clusters_good_df
    variant_results["all6_all_clusters_good_keys"] = all6_keys
    variant_results["all6_all_clusters_good_detail"] = all6_detail
    linreg_time_variant_results[variant_name] = variant_results

